In [0]:
%run ../config_setup

In [0]:
# Databricks Notebook: audit_start (parameterized — env config_setup se)
import uuid
import datetime as dt
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit

spark = SparkSession.builder.getOrCreate()

# 1. Widgets define karo
dbutils.widgets.text("job_name", "default_job", "Job Name")
dbutils.widgets.text("job_type", "PIPELINE", "Job Type")

# 2. Widget se values read karo
job_name = dbutils.widgets.get("job_name")
job_type = dbutils.widgets.get("job_type")

# 3. Run ID banao
now_utc = dt.datetime.now(dt.timezone.utc)
run_id = f"RUN-{now_utc:%Y%m%d%H%M%S}-{uuid.uuid4().hex[:6]}"

# 4. Dataframe banao
audit_data = [(run_id, job_name, job_type, "STARTED", "N")]
cols = ["run_id", "job_name", "job_type", "status", "is_completed"]

df = spark.createDataFrame(audit_data, cols) \
    .withColumn("start_ts", current_timestamp()) \
    .withColumn("end_ts", lit(None).cast("timestamp")) \
    .withColumn("error_message", lit(None).cast("string")) \
    .withColumn("execution_time_sec", lit(None).cast("double"))

# 5. Insert karo — audit table env-aware
df.write.format("delta").mode("append").saveAsTable(f"{AUDIT}.job_audit_log")

# 6. run_id aage pass karo
dbutils.jobs.taskValues.set(key="run_id", value=run_id)
print(f"✅ Audit START logged. Job: {job_name}, Run ID: {run_id}")